# Planetary Systems Architecture Analysis

This notebook reads the master catalogue and generates an HTML with a graph for each planetary system.

Each graph shows:
- X axis: `semi_major_axis` (AU).
- Habitability bands: optimistic y conservative.
- Planets as coloured dots and size as in `planet_category`.

In [9]:
from pathlib import Path
import html as html_lib

import numpy as np
import pandas as pd
import plotly.graph_objects as go

# Input / output
CATALOG_PATH = Path('Exoplanets_SolarNeighbourhood _Catalogue.csv')
HTML_OUTPUT = Path('exoplanets_system_architecture.html')

print('Input catalogue:', CATALOG_PATH)
print('HTML output:', HTML_OUTPUT)

Input catalogue: Exoplanets_SolarNeighbourhood _Catalogue.csv
HTML output: exoplanets_system_architecture.html


In [ ]:
# Visual style for planet categories (aligned with table_generator palette)
CATEGORY_STYLE = {
    'Terrestrial': {'color': '#5a3a1b', 'size': 11},
    'Sub-Neptune': {'color': '#0b4f6c', 'size': 14},
    'Ice Giant': {'color': '#1e3a8a', 'size': 18},
    'Gas Giant': {'color': '#a04d11', 'size': 22},
    'Unknown': {'color': '#495057', 'size': 10},
}

df = pd.read_csv(CATALOG_PATH)

required_cols = [
    'name',
    'star_name',
    'planet_status',
    'star_distance',
    'semi_major_axis',
    'orbital_period',
    'planet_category',
    'mass_earth_th',
    'radius_earth_th',
    'hz_inner_optimistic_au',
    'hz_inner_conservative_au',
    'hz_outer_conservative_au',
    'hz_outer_optimistic_au',
]

missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f'Columns missing in catalogue: {missing}')

# Remove discarded planets before plotting
df['planet_status'] = df['planet_status'].fillna('Unknown').astype(str).str.strip()
discarded_mask = df['planet_status'].str.lower().eq('discarded')
n_discarded = int(discarded_mask.sum())
df = df.loc[~discarded_mask].copy()

# Coerce numeric columns used for axis, HZ limits, and hover
numeric_cols = [
    'star_distance',
    'semi_major_axis',
    'orbital_period',
    'mass_earth_th',
    'radius_earth_th',
    'hz_inner_optimistic_au',
    'hz_inner_conservative_au',
    'hz_outer_conservative_au',
    'hz_outer_optimistic_au',
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Fallback system name if star_name is empty
df['system_name'] = df['star_name'].astype(str).str.strip()
fallback_mask = df['system_name'].isin(['', 'nan', 'None'])
df.loc[fallback_mask, 'system_name'] = (
    df.loc[fallback_mask, 'name']
    .astype(str)
    .str.rsplit(' ', n=1)
    .str[0]
)

df['planet_category'] = df['planet_category'].fillna('Unknown').astype(str)
df.loc[~df['planet_category'].isin(CATEGORY_STYLE), 'planet_category'] = 'Unknown'

systems = sorted(df['system_name'].dropna().unique())
print(f'Loaded {len(df)} planets in {len(systems)} systems.')
print(f'Discarded planets removed: {n_discarded}')


def _finite_values(values):
    arr = np.asarray(values, dtype=float)
    return arr[np.isfinite(arr)]



def _fmt_value(value, decimals=3):
    if pd.isna(value) or not np.isfinite(value):
        return 'Unknown'
    return f"{float(value):.{decimals}f}"



def build_system_figure(system_df, system_name, system_distance_pc):
    fig = go.Figure()

    # Use representative HZ limits (median across planets in the same system)
    rv = system_df['hz_inner_optimistic_au'].median(skipna=True)
    rg = system_df['hz_inner_conservative_au'].median(skipna=True)
    mg = system_df['hz_outer_conservative_au'].median(skipna=True)
    em = system_df['hz_outer_optimistic_au'].median(skipna=True)

    added_optimistic_hz = False
    added_conservative_hz = False

    # Optimistic HZ band
    if np.isfinite(rv) and np.isfinite(em) and em > rv:
        fig.add_vrect(
            x0=rv, x1=em,
            fillcolor='rgba(130, 200, 120, 0.20)',
            line_width=0,
            layer='below'
        )
        added_optimistic_hz = True

    # Conservative HZ band
    if np.isfinite(rg) and np.isfinite(mg) and mg > rg:
        fig.add_vrect(
            x0=rg, x1=mg,
            fillcolor='rgba(35, 130, 65, 0.33)',
            line_width=0,
            layer='below'
        )
        added_conservative_hz = True

    # Add an in-chart note when no HZ band can be shown
    if not added_optimistic_hz and not added_conservative_hz:
        fig.add_annotation(
            x=0.99,
            y=0.95,
            xref='paper',
            yref='paper',
            xanchor='right',
            yanchor='top',
            text='Missing data for HZ',
            showarrow=False,
            font=dict(size=11, color='#b45309'),
            bgcolor='rgba(255, 244, 214, 0.9)',
            bordercolor='rgba(180, 83, 9, 0.45)',
            borderwidth=1,
        )

    # Planets by category
    for category, style in CATEGORY_STYLE.items():
        cat_df = system_df[system_df['planet_category'] == category].copy()
        cat_df = cat_df[np.isfinite(cat_df['semi_major_axis'])]
        if cat_df.empty:
            continue

        status_text = cat_df['planet_status'].fillna('Unknown').astype(str).str.strip()
        status_lower = status_text.str.lower()
        is_candidate = status_lower.eq('candidate')
        status_html = np.array([
            "<span style='color:#2f9e44; font-weight:600;'>Confirmed</span>" if low == 'confirmed'
            else "<span style='color:#f2c94c; font-weight:600;'>Candidate</span>" if low == 'candidate'
            else f"<span style='color:#6c757d; font-weight:600;'>{raw}</span>"
            for raw, low in zip(status_text.to_numpy(), status_lower.to_numpy())
        ], dtype=object)

        marker_line_color = np.where(is_candidate, '#ffd43b', 'rgba(255,255,255,0.92)')
        marker_line_width = np.where(is_candidate, 2.6, 1.1)

        custom = np.column_stack([
            cat_df['mass_earth_th'].map(_fmt_value).to_numpy(),
            cat_df['radius_earth_th'].map(_fmt_value).to_numpy(),
            cat_df['orbital_period'].map(_fmt_value).to_numpy(),
            status_html,
            cat_df['planet_category'].to_numpy(),
        ])

        fig.add_trace(go.Scatter(
            x=cat_df['semi_major_axis'],
            y=np.zeros(len(cat_df)),
            mode='markers',
            name=category,
            showlegend=False,
            marker=dict(
                size=style['size'],
                color=style['color'],
                opacity=0.92,
                line=dict(width=marker_line_width, color=marker_line_color)
            ),
            text=cat_df['name'],
            customdata=custom,
            hovertemplate=(
                '<b>%{text}</b><br>'
                'Mass (M_earth): %{customdata[0]}<br>'
                'Radius (R_earth): %{customdata[1]}<br>'
                'Orbital period (d): %{customdata[2]}<br>'
                'Category: %{customdata[4]}<br>'
                'Status: %{customdata[3]}<extra></extra>'
            )
        ))

    # Dynamic x range from planets and HZ limits
    x_planets = _finite_values(system_df['semi_major_axis'])
    x_hz = _finite_values([rv, rg, mg, em])
    x_all = np.concatenate([x_planets, x_hz]) if len(x_hz) else x_planets

    if len(x_all):
        x_min = max(0.0, float(np.nanmin(x_all)) * 0.85)
        x_max = float(np.nanmax(x_all)) * 1.15
        if x_max <= x_min:
            x_max = x_min + 1.0
    else:
        x_min, x_max = 0.0, 1.0

    if np.isfinite(system_distance_pc):
        title_text = f'{system_name} (d = {system_distance_pc:.2f} pc)'
    else:
        title_text = f'{system_name} (d = Unknown pc)'

    fig.update_layout(
        title=title_text,
        template='plotly_white',
        height=260,
        margin=dict(l=38, r=28, t=52, b=36),
    )
    fig.update_xaxes(title='Semi-major axis a (AU)', range=[x_min, x_max])
    fig.update_yaxes(title='', showticklabels=False, showgrid=False, zeroline=False, range=[-0.2, 0.2])

    return fig



def build_global_legend_html():
    labels = [
        ('Terrestrial', 'Terrestrial'),
        ('Sub-Neptune', 'Sub-Neptune'),
        ('Ice Giant', 'Ice Giant'),
        ('Gas Giant', 'Gas Giant'),
        ('Unknown', 'Unknown'),
    ]
    chips = []
    for key, label in labels:
        color = CATEGORY_STYLE[key]['color']
        size = max(10, int(CATEGORY_STYLE[key]['size'] * 0.78))
        chips.append(
            f"<div class='legend-item'><span class='dot' style='background:{color}; width:{size}px; height:{size}px;'></span>{label}</div>"
        )
    return ''.join(chips)



def build_hz_legend_html():
    items = [
        ('Optimistic HZ', 'rgba(130, 200, 120, 0.20)'),
        ('Conservative HZ', 'rgba(35, 130, 65, 0.33)'),
    ]
    chips = []
    for label, color in items:
        chips.append(
            f"<div class='legend-item'><span class='hz-band' style='background:{color};'></span>{label}</div>"
        )
    return ''.join(chips)



def build_status_legend_html():
    items = [
        ('Confirmed', "<span class='dot status-confirmed'></span>"),
        ('Candidate', "<span class='dot status-candidate'></span>"),
    ]
    chips = []
    for label, marker in items:
        chips.append(f"<div class='legend-item'>{marker}{label}</div>")
    return ''.join(chips)


system_distance = (
    df.groupby('system_name', as_index=False)['star_distance']
      .median()
      .rename(columns={'star_distance': 'system_distance_pc'})
)
system_distance['sort_distance'] = system_distance['system_distance_pc'].fillna(np.inf)

systems_ordered = system_distance.sort_values(['sort_distance', 'system_name'])[['system_name', 'system_distance_pc']]

html_blocks = []
for row in systems_ordered.itertuples(index=False):
    system_name = row.system_name
    system_distance_pc = row.system_distance_pc
    sdf = df[df['system_name'] == system_name].copy()

    # Skip systems without any valid semi-major-axis values
    if not np.isfinite(pd.to_numeric(sdf['semi_major_axis'], errors='coerce')).any():
        continue

    fig = build_system_figure(sdf, system_name, system_distance_pc)
    chart_html = fig.to_html(full_html=False, include_plotlyjs='cdn')
    safe_system_name = html_lib.escape(system_name.lower(), quote=True)
    html_blocks.append(f'<div class="chart" data-system-name="{safe_system_name}">{chart_html}</div>')

if not html_blocks:
    raise RuntimeError('No systems with valid semi_major_axis were found.')

legend_html = build_global_legend_html()
hz_legend_html = build_hz_legend_html()
status_legend_html = build_status_legend_html()

html_doc = f'''<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>Planetary Systems Structure Comparison</title>
  <style>
    body {{ font-family: 'Segoe UI', Tahoma, sans-serif; margin: 20px; background: #f5f7fb; color: #1f2a37; }}
    .header {{ background: white; border-radius: 14px; padding: 16px 18px; box-shadow: 0 3px 12px rgba(20, 28, 45, 0.08); margin-bottom: 18px; }}
    .header-legends-search {{ display: flex; align-items: flex-start; justify-content: space-between; gap: 16px; margin-top: 8px; }}
    .legends-stack {{ display: flex; flex-direction: column; gap: 8px; flex: 1 1 auto; min-width: 0; }}
    .legend-main {{ display: flex; flex-wrap: nowrap; gap: 14px; min-width: 0; overflow-x: auto; }}
    .legend-main .legend-item {{ flex: 0 0 auto; }}
    .legend-title {{ font-size: 0.92rem; font-weight: 600; color: #334155; margin-top: 2px; }}
    .search-panel {{ display: flex; flex-direction: column; gap: 6px; align-items: stretch; width: 280px; flex: 0 0 280px; }}
    .search-panel label {{ font-size: 0.92rem; font-weight: 600; color: #334155; }}
    .search-panel input {{ width: 100%; box-sizing: border-box; border: 1px solid #cbd5e1; border-radius: 10px; padding: 10px 12px; font-size: 0.98rem; background: #fff; color: #1f2a37; }}
    .search-panel input:focus {{ outline: none; border-color: #0b4f6c; box-shadow: 0 0 0 3px rgba(11, 79, 108, 0.12); }}
    .search-note {{ font-size: 0.84rem; color: #64748b; }}
    h1 {{ margin: 0 0 8px 0; font-size: 1.5rem; }}
    p {{ margin: 0 0 10px 0; color: #42526a; line-height: 1.4; }}
    .legend {{ display: flex; flex-wrap: wrap; gap: 14px; margin-top: 8px; }}
    .legend-item {{ display: inline-flex; align-items: center; gap: 7px; font-size: 0.95rem; color: #1f2a37; }}
    .dot {{ display: inline-block; border-radius: 999px; border: 1px solid rgba(0,0,0,0.16); }}
    .status-confirmed {{ width: 12px; height: 12px; background: #64748b; border-color: rgba(255,255,255,0.9); }}
    .status-candidate {{ width: 12px; height: 12px; background: transparent; border: 2px solid #ffd43b; }}
    .hz-legend {{ display: flex; flex-wrap: wrap; gap: 14px; margin-top: 10px; }}
    .hz-band {{ display: inline-block; width: 22px; height: 12px; border-radius: 4px; border: 1px solid rgba(0,0,0,0.22); }}
    .chart {{
      background: white;
      border-radius: 14px;
      box-shadow: 0 3px 12px rgba(20, 28, 45, 0.08);
      padding: 6px 8px 0 8px;
      margin: 14px 0;
    }}
    .chart.is-hidden {{ display: none; }}
    .no-results {{ display: none; background: white; border-radius: 14px; box-shadow: 0 3px 12px rgba(20, 28, 45, 0.08); padding: 18px; margin: 14px 0; color: #334155; }}
  </style>
</head>
<body>
  <section class="header">
    <h1>Planetary Systems Structure Comparison</h1>
    <p>This report compares system architectures using semi-major axis (AU). Systems are ordered by stellar distance.</p>
    <div class="header-legends-search">
      <div class="legends-stack">
        <div class="legend-title">Color code:</div>
        <div class="legend-main legend">{legend_html}</div>
        <div class="legend">{status_legend_html}</div>
        <div class="hz-legend">{hz_legend_html}</div>
      </div>
      <div class="search-panel">
        <label for="system-search">Search:</label>
        <input id="system-search" type="search" placeholder="Proxima" aria-label="Search systems">
        <div class="search-note">Filters the systems below as you type.</div>
      </div>
    </div>
  </section>
  <div id="no-results" class="no-results">No systems match the current search.</div>
  {''.join(html_blocks)}
  <script>
    (function () {{
      const searchInput = document.getElementById('system-search');
      const charts = Array.from(document.querySelectorAll('.chart'));
      const noResults = document.getElementById('no-results');

      function applyFilter() {{
        const query = searchInput.value.trim().toLowerCase();
        let visibleCount = 0;

        charts.forEach((chart) => {{
          const systemName = (chart.dataset.systemName || '').toLowerCase();
          const matches = query === '' || systemName.includes(query);
          chart.classList.toggle('is-hidden', !matches);
          if (matches) {{
            visibleCount += 1;
          }}
        }});

        noResults.style.display = visibleCount === 0 ? 'block' : 'none';
      }}

      searchInput.addEventListener('input', applyFilter);
      applyFilter();
    }})();
  </script>
</body>
</html>
'''

HTML_OUTPUT.write_text(html_doc, encoding='utf-8')
print(f'HTML generated: {HTML_OUTPUT.resolve()}')
print(f'Charts generated: {len(html_blocks)}')


Loaded 123 planets in 60 systems.
Discarded planets removed: 7
HTML generated: C:\Users\Rafa\OneDrive\Escritorio\Uni\5º\Prácticas CAB\Pruebas\exoplanets_system_architecture.html
Charts generated: 60


In [11]:
# Optional: quick summary table
summary = (
    df.groupby('system_name', as_index=False)
      .agg(
          n_planets=('name', 'count'),
          min_a_au=('semi_major_axis', 'min'),
          max_a_au=('semi_major_axis', 'max')
      )
      .sort_values(['n_planets', 'system_name'], ascending=[False, True])
)
summary.head(20)

,system_name,n_planets,min_a_au,max_a_au
45,HD 219134,6,0.038760,2.82800
27,GJ 667 C,5,0.050431,0.54900
3,Barnard's star,4,0.018800,0.03810
35,GJ 876,4,0.021000,0.34500
36,GJ 887,4,0.041700,0.21200
50,Luyten's Star,4,0.036467,0.84900
59,tau Cet,4,0.133000,1.33400
0,61 Vir,3,0.049310,0.46770
1,82 Eri,3,0.125700,1.35410
2,AU Mic,3,0.070000,0.11900
